<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/course_project_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to HLT Project (Template)

- Student(s) Name(s): Lauris Seebeck
- Date: 14.6.2025
- Chosen Corpus: IMDB
- Contributions (if group project): -

### Corpus information

- Description of the chosen corpus: The chosen corpus is the given huggingface IMDB dataset.
- Paper(s) and other published materials related to the corpus: Sentiment Analysis Using Machine Learning Techniques on IMDB Dataset (Selen Nazli Başa and Muhammet Sinan Basarslan)
- State-of-the-art performance (best published results) on this corpus: 90% (from the research above)

---

## 1. Setup

In [1]:
import datasets
import torch
import torch.nn as nn
import transformers
import tensorflow as tf
import numpy as np
import evaluate

from transformers import TrainingArguments, Trainer, AutoTokenizer
from pprint import pprint
from keras import layers

/home/lauris/.conda/envs/human-language-technology/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-15 23:33:28.052625: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-15 23:33:28.158835: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---

## 2. Data download and preprocessing

### 2.1. Download the corpus

In [2]:
# Loading the dataset

DATASET = 'imdb'

dataset = datasets.load_dataset(DATASET)

### 2.2. Preprocessing

In [3]:
# Testing if dataset works:

pprint(dataset)

raw_train_dset = dataset['train']
raw_test_dset = dataset['test']

print(raw_train_dset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


### Prepping dataset:

In [4]:
max_features = 20000
sequence_length = 512

# Creating a function for layer vectorization:
vectorize_layer = layers.TextVectorization(
    standardize=None,
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

In [5]:
# Vectorizing the text

train_text = raw_train_dset['text']
print(train_text[0])

vectorize_layer.adapt(train_text)
print(train_text[0])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [6]:
# Function for text vectorization:

def vectorize_text(dataset):
    text = tf.expand_dims(dataset['text'], -1)
    label = dataset['label']
    vectorized_text = vectorize_layer(text)
    return {'input_ids': vectorized_text, 'label': label}

In [7]:
#testing if vectorization works:
print(vectorize_layer.get_vocabulary()[313])
print(vectorize_layer.get_vocabulary()[1287])

completely
convincing


In [8]:
# Vectorizing both train and test datasets keeping the original text, labels and adding input ids, which is the vectorized text:

train_dset = raw_train_dset.map(vectorize_text, batched=True)

print(train_dset)
print(train_dset[0])

Map: 100%|██████████| 25000/25000 [00:00<00:00, 27998.49 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids'],
    num_rows: 25000
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this wa

In [9]:
test_dset = raw_test_dset.map(vectorize_text, batched=True)

Map: 100%|██████████| 25000/25000 [00:00<00:00, 29533.85 examples/s]


In [10]:
# setting up evaluation measurement:

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids
    preds = np.argmax(predictions, axis=-1)
    acc = np.mean(preds == labels)
    return {"accuracy": acc}

In [11]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

/home/lauris/.conda/envs/human-language-technology/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [12]:
# Data collection function (used the same function from task05):

def data_collator(list_of_examples):
    batch={"labels":torch.tensor(list(ex["label"] for ex in list_of_examples))}
    tensors=[]
    max_len=max(len(example["input_ids"]) for example in list_of_examples)
    for example in list_of_examples:
        ids=torch.tensor(example["input_ids"])
        padded=nn.functional.pad(ids,(0,max_len-ids.shape[0]))
        tensors.append(padded)
    batch["input_ids"]=torch.vstack(tensors)
    return batch

---

## 3. Machine learning model

### 3.1. Model training

In [13]:
batch_size = 8
epochs = 5
print(epochs)


5


In [14]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [15]:
# model was inspired from task05:

class MLPConfig(transformers.PretrainedConfig):
  pass

class MLP(transformers.PreTrainedModel):
    config_class = MLPConfig

    def __init__(self, config):
        super().__init__(config)
        self.vocab_size=config.vocab_size
        self.embedding = nn.Embedding(
            num_embeddings=config.vocab_size + 1,
            embedding_dim=config.hidden_size,
            padding_idx=0
        )
        nn.init.uniform_(self.embedding.weight.data,-0.001,0.001)
        self.output=nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)

    def forward(self, input_ids, labels):
        embedded = self.embedding(input_ids)
        embedded_summed = torch.sum(embedded, dim=1)
        logits = self.output(embedded_summed)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}


In [16]:
config = MLPConfig(vocab_size=tokenizer.vocab_size, hidden_size=512, nlabels=2)
model = MLP(config)

### 3.2 Hyperparameter optimization

In [17]:
training_args = TrainingArguments(
    output_dir="model",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dset,
    eval_dataset=test_dset.select(range(1000)),
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [19]:
#training the model
trainer.train()

  3%|▎         | 501/15625 [00:47<23:52, 10.55it/s]

{'loss': 0.6212, 'grad_norm': 2.5127131938934326, 'learning_rate': 1.936e-05, 'epoch': 0.16}


  6%|▋         | 1002/15625 [01:36<23:35, 10.33it/s]

{'loss': 0.5066, 'grad_norm': 6.369744777679443, 'learning_rate': 1.8720000000000004e-05, 'epoch': 0.32}


 10%|▉         | 1502/15625 [02:27<23:13, 10.13it/s]

...

                                                     
100%|██████████| 15625/15625 [26:49<00:00,  9.32it/s]

{'eval_loss': 0.31928780674934387, 'eval_accuracy': 0.879, 'eval_runtime': 0.4988, 'eval_samples_per_second': 2004.985, 'eval_steps_per_second': 250.623, 'epoch': 5.0}


100%|██████████| 15625/15625 [26:50<00:00,  9.70it/s]

{'train_runtime': 1610.1211, 'train_samples_per_second': 77.634, 'train_steps_per_second': 9.704, 'train_loss': 0.2528537556152344, 'epoch': 5.0}


TrainOutput(global_step=15625, training_loss=0.2528537556152344, metrics={'train_runtime': 1610.1211, 'train_samples_per_second': 77.634, 'train_steps_per_second': 9.704, 'total_flos': 393984000000.0, 'train_loss': 0.2528537556152344, 'epoch': 5.0})

### 3.3. Evaluation on test set

In [20]:
eval_results = trainer.evaluate(test_dset)

print(eval_results)

100%|██████████| 3125/3125 [00:13<00:00, 232.19it/s]

{'eval_loss': 0.3116682171821594, 'eval_accuracy': 0.87568, 'eval_runtime': 13.472, 'eval_samples_per_second': 1855.701, 'eval_steps_per_second': 231.963, 'epoch': 5.0}


---

## 4. Results and summary

### 4.1 Corpus insights

The corpus includes 12500 positive and 12500 negative labeled text with labels 0: negative and 1: positive.
The corpus includes 25000 rows of train, test and unsupervised datasets.

### 4.2 Results

I tried with multiple different hyperparameters, and I got around 85-87% everytime. 

### 4.3 Relation to state of the art

Looking at the hugging face community posts, one of the best recorded accuracies are around 0.95.

---

## 5. Bonus Task (optional)

### 5.1. Annotating out-of-domain documents

(Briefly describe the chosen out-of-domain documents)

(Briefly describe the process of annotation)

### 5.2 Conversion into dataset

In [21]:
# Your code to convert the annotations into a dataset here

### 5.3. Model evaluation on out-of-domain test set

In [22]:
# Your code to evaluate the model on the out-of-domain test set here

### 5.4 Bonus task results

(Present the results of the evaluation on the out-of-domain test set)

### 5.5. Annotated data

In [23]:
# Include your annotated out-of-domain data here